In [0]:
silver_drivers = spark.table(
    "workspace.transportation_analytics.silver_drivers"
)

silver_driver_metrics = spark.table(
    "workspace.transportation_analytics.silver_driver_monthly_metrics"
)

print("Silver driver tables loaded successfully")

silver_drivers.printSchema()
silver_driver_metrics.printSchema()

In [0]:
from pyspark.sql import functions as F

In [0]:
gold_driver_performance = (
    silver_driver_metrics
    .join(
        silver_drivers,
        on="driver_id",
        how="left"
    )
)

print("Driver data joined successfully")

In [0]:
gold_driver_performance = (
    gold_driver_performance
    .groupBy(
        "driver_id",
        "first_name",
        "last_name",
        "employment_status",
        "years_experience"
    )
    .agg(
        F.sum("trips_completed").alias("total_trips"),
        F.sum("total_miles").alias("total_miles"),
        F.sum("total_revenue").alias("total_revenue"),
        F.avg("on_time_delivery_rate").alias("avg_on_time_delivery_rate"),
        F.sum("total_miles").alias("miles_for_mpg"),
        F.sum("total_fuel_gallons").alias("total_fuel_gallons"),
        F.avg("average_idle_hours").alias("avg_idle_hours")
    )
)

print("Driver performance calculated successfully")

In [0]:
gold_driver_performance = (
    gold_driver_performance
    .withColumn(
        "overall_mpg",
        F.round(
            F.col("miles_for_mpg") / F.col("total_fuel_gallons"),
            2
        )
    )
    .withColumn(
        "revenue_per_mile",
        F.round(
            F.col("total_revenue") / F.col("total_miles"),
            2
        )
)
)

print("MPG and revenue per mile calculated successfully")

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_driver_performance"
)

target.alias("t").merge(
    gold_driver_performance.alias("s"),
    "t.driver_id = s.driver_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Driver Performance table updated using MERGE")

In [0]:
spark.table(
    "workspace.transportation_analytics.gold_driver_performance"
).show(20, truncate=False)

In [0]:
display(
    gold_driver_performance
    .orderBy(F.desc("total_revenue"))
    .select(
        "driver_id",
        "first_name",
        "last_name",
        "total_trips",
        "total_miles",
        "total_revenue",
        "avg_on_time_delivery_rate",
        "overall_mpg",
        "revenue_per_mile"
    )
)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.